# 02 — Protocol comparison and error analysis

**Purpose.** Quantify what notebook 01 predicted: run the same models under a random split and
under a chronological split, measure how much of the pooled score is calendar signal, and establish
what the model is actually worth to a campaign.

**How to read this notebook.** Every model is built and evaluated by the production pipeline
(`term_deposit.pipelines.experiment.run_experiment`), the same function `scripts/train.py` calls.
Nothing here reimplements training or scoring, so the numbers below are the numbers the pipeline
writes to `reports/metrics/`.

**Prerequisite.** Run `make data` once. This notebook trains models, so allow a few minutes.

**Outline**
1. Setup
2. Protocol A — random split
3. Protocol B — out-of-time split
4. The gap: pooled versus within-month
5. Protocol C — rolling-origin backtest, and model selection
6. Does the macro block earn its place?
7. Calibration under a regime change
8. What drives the ranking
9. Error analysis: who the model gets wrong
10. Conclusions

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from term_deposit import constants
from term_deposit.config import load_config
from term_deposit.evaluation import plots
from term_deposit.evaluation.explain import permutation_feature_importance
from term_deposit.evaluation.metrics import binary_metrics, lift_at_k
from term_deposit.pipelines.experiment import prepare_dataset, run_experiment
from term_deposit.utils.logging import configure_logging

configure_logging("WARNING", force=True)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
plt.rcParams["figure.dpi"] = 110

ROOT = Path.cwd().parents[1] if Path.cwd().name == "exploratory" else Path.cwd()
CONFIGS = [ROOT / "configs" / "base.yaml", ROOT / "configs" / "training.yaml"]

# The dataset is loaded once and reused by both protocols, so the only thing that differs
# between the two runs is the split.
base = load_config(CONFIGS, project_root=ROOT)
prepared = prepare_dataset(base, download=False)
print(f"{len(prepared.frame):,} rows across {prepared.period_key.nunique()} contact months")

## 1. Setup

Five candidates are configured in `configs/training.yaml`, including a `DummyClassifier(strategy=
"prior")` baseline. The baseline is not a formality: it makes the no-skill reference executable, so
any claim that a model is useful has to clear a line that is printed in the same table.

Both runs below use `persist=False` — they compute and report but do not overwrite the artifacts
that `scripts/train.py` produced.

In [ ]:
def run(strategy: str, **overrides) -> object:
    """Run one experiment under a named split strategy."""
    config = load_config(
        CONFIGS,
        overrides=[f"split.strategy={strategy}", *[f"{k}={v}" for k, v in overrides.items()]],
        project_root=ROOT,
    )
    return run_experiment(config, data=prepared, persist=False)


DISPLAY = [
    "model", "base_rate", "average_precision", "roc_auc", "within_period_roc_auc",
    "roc_auc_inflation", "precision_at_0.20", "lift_at_0.20", "brier_score", "ece",
]

for spec in load_config(CONFIGS, project_root=ROOT).require_training().enabled_models:
    print(f"{spec.name:22s} {spec.estimator:20s} balance={spec.balance_strategy}")

## 2. Protocol A — random split

The protocol the original analysis used, and the one almost every published result on this dataset
uses. Rows are shuffled and stratified, so every calendar month appears on both sides of the split.

In [ ]:
random_result = run("random")

print(f"sizes        {random_result.split.sizes}")
print("base rates   " + ", ".join(f"{k}={v:.4f}" for k, v in
                                  random_result.split.positive_rates.items()))
random_result.comparison[DISPLAY].round(4)

Pooled ROC-AUC lands around 0.80–0.81 for all four real models, reproducing the familiar result.

Now read the two columns next to it. `within_period_roc_auc` is the same models, scoring the same
test rows, but with the metric computed separately inside each contact month and then averaged by
row count. `roc_auc_inflation` is the difference.

In [ ]:
gap = random_result.comparison[
    ["model", "roc_auc", "within_period_roc_auc", "roc_auc_inflation"]
].round(4)

fig, ax = plt.subplots(figsize=(8, 4.5))
positions = np.arange(len(gap))
ax.barh(positions + 0.2, gap["roc_auc"], 0.38, label="pooled ROC-AUC", color="#4C78A8")
ax.barh(positions - 0.2, gap["within_period_roc_auc"], 0.38, label="within-month ROC-AUC",
        color="#F58518")
ax.axvline(0.5, color="k", linestyle="--", linewidth=1, label="random ranking")
ax.set(yticks=positions, xlim=(0.45, 0.9), xlabel="ROC-AUC",
       title="Random split: how much of the score survives holding the calendar fixed")
ax.set_yticklabels(gap["model"])
ax.legend(fontsize=8, loc="lower right")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

gap

Roughly 0.22 ROC-AUC points — about half the distance from random guessing to the reported score —
comes from separating calendar months rather than customers.

The per-month breakdown makes the mechanism concrete. The large 2008 months, which carry most of the
data, sit barely above chance.

In [ ]:
best_random = max(random_result.reports,
                  key=lambda r: r.test_metrics.average_precision)
per_period = pd.DataFrame(best_random.within_period["by_period"])

print(f"model: {best_random.model_name}")
print(f"pooled ROC-AUC          {best_random.within_period['pooled_roc_auc']:.4f}")
print(f"within-month (weighted) {best_random.within_period['weighted_roc_auc']:.4f}")
print(f"correlation between a month's base rate and its mean predicted score: "
      f"{np.corrcoef(per_period['base_rate'], per_period['mean_score'])[0, 1]:.4f}")

per_period[["period", "n_rows", "base_rate", "mean_score", "roc_auc", "lift_at_20pct"]].round(4)

That last correlation is the leak stated in one number: the model's average output for a month
tracks that month's actual conversion rate almost perfectly. It has learned the calendar.

## 3. Protocol B — out-of-time split

Train on the earliest months, calibrate and choose thresholds on the next block, test on the most
recent. No test row is contemporaneous with any training row.

The base rates make the difficulty visible: the model is fitted where 6.7% of contacts convert and
scored where 52.1% do.

In [ ]:
oot_result = run("out_of_time")

print(f"sizes        {oot_result.split.sizes}")
print("base rates   " + ", ".join(f"{k}={v:.4f}" for k, v in
                                  oot_result.split.positive_rates.items()))
print(f"boundaries   {oot_result.split.boundaries}")
oot_result.comparison[DISPLAY].round(4)

Two things to note, and neither is comfortable.

First, **average precision is not comparable across the two tables.** No-skill AP equals the base
rate. Protocol A's 0.47 sits against a floor of 0.11; protocol B's 0.68 sits against a floor of 0.52.
The second is the weaker result despite the larger number. Always read AP against the base rate in
its own row — which is why the pipeline prints them side by side.

Second, the ranking of models has changed. XGBoost, essentially tied for first under the random
split, is last here.

In [ ]:
comparison = (
    random_result.comparison[["model", "average_precision", "roc_auc", "lift_at_0.20"]]
    .rename(columns=lambda c: f"random_{c}" if c != "model" else c)
    .merge(
        oot_result.comparison[["model", "average_precision", "roc_auc", "lift_at_0.20"]]
        .rename(columns=lambda c: f"oot_{c}" if c != "model" else c),
        on="model",
    )
)
comparison["ap_over_baseline_random"] = (
    comparison["random_average_precision"] - random_result.split.positive_rates["test"]
)
comparison["ap_over_baseline_oot"] = (
    comparison["oot_average_precision"] - oot_result.split.positive_rates["test"]
)
comparison.round(4)

Measured as lift over the no-skill floor, every model loses roughly two thirds of its apparent
value when the evaluation stops sharing months across the split.

## 4. The gap, side by side

One chart, both protocols, for the model that is ultimately shipped.

In [ ]:
rows = []
for label, result in (("random", random_result), ("out_of_time", oot_result)):
    for report in result.reports:
        rows.append({
            "protocol": label,
            "model": report.model_name,
            "base_rate": report.test_metrics.base_rate,
            "average_precision": report.test_metrics.average_precision,
            "ap_lift_over_no_skill": (
                report.test_metrics.average_precision / report.test_metrics.base_rate
            ),
            "lift_at_20pct": report.test_metrics.top_k["0.20"]["lift"],
            "roc_auc": report.test_metrics.roc_auc,
            "within_period_roc_auc": report.within_period.get("weighted_roc_auc", np.nan),
        })
protocol_table = pd.DataFrame(rows)

fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4.5))
for axis, metric, title in (
    (left, "ap_lift_over_no_skill", "Average precision relative to no-skill"),
    (right, "lift_at_20pct", "Lift at 20% capacity"),
):
    pivot = protocol_table.pivot(index="model", columns="protocol", values=metric)
    pivot.plot(kind="bar", ax=axis, color=["#F58518", "#4C78A8"], width=0.75)
    axis.axhline(1.0, color="k", linestyle="--", linewidth=1)
    axis.set(title=title, xlabel="")
    axis.tick_params(axis="x", rotation=25)
    axis.grid(axis="y", alpha=0.3)
    axis.legend(fontsize=8)
plt.tight_layout()
plt.show()

protocol_table.round(4)

## 5. Protocol C — rolling-origin backtest

A single held-out window is one draw from a noisy distribution, and the out-of-time test set here is
only about 2,000 rows. The backtest is the more defensible estimate: for each of the last nine
months, retrain on everything before it and score that month. That is what monthly retraining in
production would actually look like.

Model selection uses the mean of this, not the single window above.

In [ ]:
backtest_rows = [
    {"model": report.model_name, **fold}
    for report in oot_result.reports
    if report.backtest
    for fold in report.backtest["folds"]
]
backtest = pd.DataFrame(backtest_rows)

summary = (
    backtest.groupby("model", observed=True)
    .agg(
        folds=("roc_auc", "size"),
        base_rate=("base_rate", "mean"),
        ap_mean=("average_precision", "mean"),
        ap_std=("average_precision", "std"),
        roc_auc_mean=("roc_auc", "mean"),
        roc_auc_std=("roc_auc", "std"),
        lift20_mean=("lift_at_20pct", "mean"),
    )
    .sort_values("ap_mean", ascending=False)
)
summary.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for name, group in backtest.groupby("model", observed=True):
    ordered = group.sort_values("period")
    style = {"linewidth": 2.5, "marker": "o"} if name == "random_forest" else {"alpha": 0.6}
    ax.plot(ordered["period"].astype(str), ordered["average_precision"], label=name, **style)

reference = backtest.drop_duplicates("period").sort_values("period")
ax.plot(reference["period"].astype(str), reference["base_rate"], "k--", linewidth=1.5,
        label="base rate (no-skill AP)")
ax.set(xlabel="test month", ylabel="average precision",
       title="Rolling-origin backtest: train on all prior months, score the next")
ax.tick_params(axis="x", rotation=45)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"selected model: {oot_result.best.model_name}")
print(f"decision threshold {oot_result.best.threshold.threshold:.4f} "
      f"(objective={oot_result.best.threshold.objective}, "
      f"chosen on {oot_result.best.threshold.chosen_on})")

The spread across months is wider than the spread across models. That is the honest reading: month
selection matters more than estimator selection on this data, and any claim that one algorithm
clearly beats another here would be over-reading the evidence.

**The operational number is the lift.** Working the top 20% of the ranked list reaches roughly 1.5×
as many subscribers as calling the same number of people at random. Useful, and far below what the
random-split lift of 3.3 suggested.

## 6. Does the macro block earn its place?

Notebook 01 argued the macro features cannot help rank customers within a batch. Test it directly:
retrain with `feature_set=client_only`, which drops all five, and compare.

In [ ]:
client_only = run("out_of_time", **{"features.feature_set": "client_only"})

macro_effect = (
    oot_result.comparison[["model", "average_precision", "roc_auc", "lift_at_0.20"]]
    .rename(columns=lambda c: f"all_{c}" if c != "model" else c)
    .merge(
        client_only.comparison[["model", "average_precision", "roc_auc", "lift_at_0.20"]]
        .rename(columns=lambda c: f"client_{c}" if c != "model" else c),
        on="model",
    )
)
macro_effect["ap_difference"] = (
    macro_effect["all_average_precision"] - macro_effect["client_average_precision"]
)
macro_effect.round(4)

Whatever the difference turns out to be on this run, the interpretation is bounded by what notebook
01 established: within a single scored batch these features are constant, so any advantage they show
here comes from the model using them to place the *whole batch* on a calibration curve, not to order
customers inside it.

That suggests a cleaner division of labour than "drop them" or "keep them":

- the macro block is information about **when to run a campaign**;
- the client and contact-history block is information about **whom to call**.

The repository keeps both feature sets available so the distinction stays measurable.

## 7. Calibration under a regime change

Ranking is one thing; probabilities are another. Any expected-value decision needs the numbers to
mean what they say.

The calibrator is fitted on the validation window (2009-06..2009-12, base rate 39%) and applied to a
test window at 52%. Watch what happens.

In [ ]:
best_oot = next(r for r in oot_result.reports if r.model_name == oot_result.best.model_name)

print(f"{'protocol':<14}{'Brier':>9}{'ECE':>9}")
print(f"{'random':<14}{best_random.test_metrics.brier_score:>9.4f}"
      f"{best_random.test_metrics.expected_calibration_error:>9.4f}")
print(f"{'out_of_time':<14}{best_oot.test_metrics.brier_score:>9.4f}"
      f"{best_oot.test_metrics.expected_calibration_error:>9.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for axis, (label, report) in zip(
    axes, (("random split", best_random), ("out-of-time split", best_oot)), strict=True
):
    bins = report.calibration
    axis.plot(bins["mean_predicted"], bins["observed_rate"], marker="o", color="#4C78A8")
    axis.plot([0, 1], [0, 1], "k--", linewidth=1)
    axis.set(xlabel="mean predicted probability", ylabel="observed rate",
             title=f"{label}\nECE={report.test_metrics.expected_calibration_error:.4f}",
             xlim=(0, 1), ylim=(0, 1))
    axis.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Calibration collapses out-of-time: the curve sits well above the diagonal because a model fitted on
a low-conversion regime systematically under-predicts in a high-conversion one.

The practical consequence is stated plainly in the model card: **out-of-time probabilities should be
treated as scores, not as rates**, until they are recalibrated on current data. Ranking transfers
across the regime change; the probability scale does not.

## 8. What drives the ranking

Permutation importance on the held-out test split, measured against average precision — the metric
that is actually being optimised. This is preferred over impurity-based importance, which is biased
towards continuous, high-cardinality features. Permuting *raw* columns rather than one-hot outputs
keeps each categorical variable's contribution in one place.

In [ ]:
winner = next(m for m in oot_result.models if m.name == oot_result.best.model_name)

importance = permutation_feature_importance(
    winner.pipeline,
    oot_result.split.X_test,
    oot_result.split.y_test,
    scoring="average_precision",
    n_repeats=10,
    random_state=42,
)

macro = set(constants.MACRO_FEATURES)
colours = ["#E45756" if f in macro else "#4C78A8" for f in importance["feature"]]

fig, ax = plt.subplots(figsize=(8, 6))
top = importance.head(15).iloc[::-1]
ax.barh(top["feature"], top["importance"],
        xerr=top["importance_std"],
        color=[c for f, c in zip(importance["feature"], colours, strict=True)
               if f in set(top["feature"])][::-1])
ax.set(xlabel="drop in average precision when permuted",
       title="Permutation importance on the out-of-time test split\n(red = macro block)")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

importance.head(15).round(4)

## 9. Error analysis: who the model gets wrong

Where does the ranking fail? Split the test set into deciles of predicted score and look at the
realised conversion rate in each — the practical question being "if we work the list from the top,
where does it stop being worth the call?"

Then look at the customers the model was most confident about and got wrong.

In [ ]:
scores = winner.predict_proba(oot_result.split.X_test)
analysis = oot_result.split.X_test.copy()
analysis["score"] = scores
analysis["actual"] = oot_result.split.y_test.to_numpy()
analysis["decile"] = pd.qcut(analysis["score"].rank(method="first", ascending=False),
                             10, labels=range(1, 11)).astype(int)

deciles = analysis.groupby("decile", observed=True).agg(
    n=("actual", "size"),
    conversion_rate=("actual", "mean"),
    mean_score=("score", "mean"),
)
deciles["lift"] = deciles["conversion_rate"] / analysis["actual"].mean()
deciles["cumulative_recall"] = (
    (deciles["n"] * deciles["conversion_rate"]).cumsum() / analysis["actual"].sum()
)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(deciles.index, deciles["conversion_rate"], color="#4C78A8", label="conversion rate")
ax.axhline(analysis["actual"].mean(), color="k", linestyle="--", linewidth=1,
           label=f"base rate ({analysis['actual'].mean():.3f})")
ax.set(xlabel="score decile (1 = highest)", ylabel="conversion rate",
       title="Decile analysis on the out-of-time test window")
ax.legend(fontsize=8)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

deciles.round(4)

In [ ]:
# The most confident mistakes, in both directions.
missed = analysis[analysis["actual"] == 1].nsmallest(5, "score")
wasted = analysis[analysis["actual"] == 0].nlargest(5, "score")

columns = ["score", "age", "job", "education", "contact", "month", "campaign",
           "pdays", "previous", "poutcome"]

print("Subscribers the model ranked lowest (missed opportunities):")
display(missed[columns].round(4))
print("\nNon-subscribers the model ranked highest (wasted calls):")
display(wasted[columns].round(4))

In [ ]:
# Where are the errors concentrated? Compare conversion rate against predicted rate by segment.
segments = []
for column in ("poutcome", "contact", "job", "education"):
    grouped = analysis.groupby(column, observed=True).agg(
        n=("actual", "size"), actual_rate=("actual", "mean"), mean_score=("score", "mean")
    )
    grouped = grouped[grouped["n"] >= 30]
    grouped["calibration_gap"] = grouped["mean_score"] - grouped["actual_rate"]
    grouped["segment_of"] = column
    segments.append(grouped.reset_index().rename(columns={column: "segment"}))

pd.concat(segments).sort_values("calibration_gap").round(4)

Segments where `calibration_gap` is strongly negative are ones the model under-scores relative to
their realised conversion — candidates for the next round of feature work, and a reminder that a
single global calibration curve is doing a lot of averaging.

## 10. Conclusions

1. **The familiar 0.81 ROC-AUC on this dataset is largely a split artefact.** Holding the calendar
   fixed, the same model on the same rows scores close to 0.59. Roughly 0.22 of the pooled figure is
   the model recognising which month a row came from.

2. **The mechanism is specific and checkable.** Four of the five macro features are exactly constant
   within a contact month; the monthly base rate moves from about 3% to about 57%. A random split
   puts the same months on both sides and rewards learning that mapping.

3. **The deployable performance is smaller and still real.** Under monthly retraining, working the
   top 20% of the list reaches about 1.5× as many subscribers as random calling — against about 3.3×
   implied by the random-split evaluation.

4. **Model choice matters less than protocol choice.** The four candidates sit within a few points of
   each other on the backtest, and the month-to-month spread is wider than the gap between them.
   Random Forest is shipped on backtest average precision, not because it is fundamentally better.

5. **Ranking transfers across the regime change; probabilities do not.** Calibration fitted on 2009
   is badly wrong on 2010, so the out-of-time probabilities are usable as an ordering and not as
   rates.

6. **The macro block answers a different question.** It carries information about *when* conditions
   favour a campaign, not about *whom* to call within one. Treating those as one modelling problem is
   what produced the inflated headline in the first place.

### What would change these conclusions

- A contemporary dataset, where the macro regime is stable across the training window, would
  weaken the leak and let the pooled metric mean more.
- Per-period recalibration, or a model given the base rate explicitly, would separate the
  calendar effect from the customer effect more cleanly than dropping features does.
- A capacity schedule and real campaign economics would replace the placeholder cost figures and
  make the threshold a genuine business decision rather than an illustration.

---

Full reasoning: [`docs/methodology.md`](../../docs/methodology.md).
Limitations and intended use: [`docs/model-card.md`](../../docs/model-card.md).